In [1]:
import numpy as np
import os

# ── point to a FILE, not a folder ─────────────────────────────────────────────
PATH = r"N:\Final preparation vedio labeling\final codes\manual_frame_passing\demo\normal_023_Person0_win0.npy"

data = np.load(PATH)

print(f"File     : {os.path.basename(PATH)}")
print(f"Shape    : {data.shape}")
print(f"Dtype    : {data.dtype}")
print(f"Min      : {data.min():.4f}")
print(f"Max      : {data.max():.4f}")
print(f"Mean     : {data.mean():.4f}")
print(f"\nFull array:\n{data}")

File     : normal_023_Person0_win0.npy
Shape    : (30, 51)
Dtype    : float32
Min      : 0.0780
Max      : 0.9977
Mean     : 0.6605

Full array:
[[0.4697126  0.61621153 0.2863814  ... 0.50077343 0.9822882  0.8679492 ]
 [0.4688982  0.61538506 0.27765566 ... 0.49929348 0.9796211  0.8629767 ]
 [0.47024748 0.6147051  0.29825044 ... 0.4995991  0.9750406  0.859448  ]
 ...
 [0.41120672 0.6123261  0.8536827  ... 0.50039667 0.9776947  0.86778235]
 [0.4110794  0.61589223 0.8452875  ... 0.49894986 0.9749611  0.8626634 ]
 [0.41127795 0.6159985  0.8479575  ... 0.49696267 0.9738518  0.86498934]]


In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)


# paths
TRAIN_DIR = r"N:\Final Preparation\FULL_DATASET\new\training"
VAL_DIR   = r"N:\Final Preparation\FULL_DATASET\new\validation"

CSV_FILE        = "labels.csv"
POSITIVE_FOLDER = r"positive\lstm_dataset"
NEGATIVE_FOLDER = r"negative\lstm_dataset"

WINDOW_SIZE  = 30
KEYPOINT_DIM = 51

HIDDEN_SIZE    = 48
NUM_LAYERS     = 1
DROPOUT        = 0.6

EPOCHS         = 100
BATCH_SIZE     = 6

LEARNING_RATE  = 0.0005
WEIGHT_DECAY   = 5e-4
CLIP_GRAD_NORM = 1.0
EARLY_STOP_PAT = 20
WARMUP_EPOCHS  = 5

AUGMENT        = True
NOISE_STD      = 0.01
SCALE_RANGE    = (0.95, 1.05)
TIME_SHIFT_MAX = 3

MODEL_SAVE_PATH = r"N:\Final Preparation\model\best_theft_lstm_model_v6.pth"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


def augment_sequence(seq):
    seq = seq.copy()

    if AUGMENT and np.random.rand() < 0.5:
        noise     = np.random.normal(0, NOISE_STD, seq.shape).astype(np.float32)
        conf_mask = np.array([(i % 3 == 2) for i in range(seq.shape[1])])
        noise[:, conf_mask] = 0.0
        seq = np.clip(seq + noise, 0.0, 1.0)

    if AUGMENT and np.random.rand() < 0.5:
        scale   = np.random.uniform(*SCALE_RANGE)
        xy_mask = ~np.array([(i % 3 == 2) for i in range(seq.shape[1])])
        seq[:, xy_mask] = np.clip(seq[:, xy_mask] * scale, 0.0, 1.0)

    if AUGMENT and np.random.rand() < 0.5:
        shift = np.random.randint(1, TIME_SHIFT_MAX + 1)
        seq   = np.roll(seq, shift, axis=0)

    return seq


class PoseSequenceDataset(Dataset):

    def __init__(self, root_dir, augment=False):
        csv_path = os.path.join(root_dir, CSV_FILE)
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"labels.csv not found: {csv_path}")

        self.df      = pd.read_csv(csv_path)
        self.root    = root_dir
        self.augment = augment

        assert "filename" in self.df.columns
        assert "label"    in self.df.columns

        pos = (self.df["label"] == 1).sum()
        neg = (self.df["label"] == 0).sum()
        print(f"  {root_dir}  ->  {len(self.df)} samples  |  theft: {pos}  normal: {neg}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        fname     = row["filename"]
        label     = int(row["label"])
        subfolder = POSITIVE_FOLDER if label == 1 else NEGATIVE_FOLDER
        path      = os.path.join(self.root, subfolder, fname)

        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")

        data = np.load(path)

        if data.shape != (WINDOW_SIZE, KEYPOINT_DIM):
            raise ValueError(f"Shape mismatch: {fname} is {data.shape}, expected ({WINDOW_SIZE}, {KEYPOINT_DIM})")

        if self.augment:
            data = augment_sequence(data)

        return (
            torch.tensor(data,  dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )


class TheftDetectionLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.lstm    = nn.LSTM(input_size=KEYPOINT_DIM, hidden_size=HIDDEN_SIZE,
                               num_layers=NUM_LAYERS, batch_first=True, dropout=0.0)
        self.bn      = nn.BatchNorm1d(HIDDEN_SIZE)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        return self.fc(out)


def get_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def evaluate(model, loader, criterion, threshold=0.50):
    model.eval()
    total_loss = 0.0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.unsqueeze(1).to(DEVICE)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            total_loss += loss.item()
            probs = torch.sigmoid(logits).squeeze(1)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.squeeze(1).long().cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs >= threshold).astype(int)

    return (
        total_loss / len(loader),
        accuracy_score (all_labels, preds),
        precision_score(all_labels, preds, zero_division=0),
        recall_score   (all_labels, preds, zero_division=0),
        f1_score       (all_labels, preds, zero_division=0),
        preds, all_labels, all_probs,
    )


def find_best_threshold(probs, labels):
    best_thresh = 0.50
    best_f1     = 0.0

    print("\nThreshold sweep:")
    print(f"  {'Thresh':>7} | {'Prec':>6} | {'Rec':>6} | {'F1':>6} | {'Acc':>6} | {'FN':>4} | {'FP':>4}")
    print("  " + "-" * 55)

    for t in np.arange(0.30, 0.81, 0.05):
        preds  = (probs >= t).astype(int)
        prec   = precision_score(labels, preds, zero_division=0)
        rec    = recall_score   (labels, preds, zero_division=0)
        f1     = f1_score       (labels, preds, zero_division=0)
        acc    = accuracy_score (labels, preds)
        cm     = confusion_matrix(labels, preds)
        marker = "  <- best" if f1 > best_f1 else ""
        print(f"  {t:>7.2f} | {prec:>5.1%} | {rec:>5.1%} | {f1:>5.1%} | {acc:>5.1%} | {cm[1,0]:>4} | {cm[0,1]:>4}{marker}")
        if f1 > best_f1:
            best_f1     = f1
            best_thresh = round(float(t), 2)

    high_recall_thresh = best_thresh
    for t in np.arange(0.30, 0.81, 0.05):
        rec = recall_score(labels, (probs >= t).astype(int), zero_division=0)
        if rec >= 0.90:
            high_recall_thresh = round(float(t), 2)
            break

    print(f"\n  best F1 threshold    : {best_thresh:.2f}  (F1={best_f1:.4f})")
    print(f"  high-recall threshold: {high_recall_thresh:.2f}  (catches >=90% of thefts)")
    return best_thresh, best_f1


def train_model():
    params = sum(p.numel() for p in TheftDetectionLSTM().parameters() if p.requires_grad)
    print(f"Device: {DEVICE}  |  params: {params:,}  |  window: {WINDOW_SIZE}")

    print("\nLoading data...")
    train_dataset = PoseSequenceDataset(TRAIN_DIR, augment=True)
    val_dataset   = PoseSequenceDataset(VAL_DIR,   augment=False)

    num_pos    = (train_dataset.df["label"] == 1).sum()
    num_neg    = (train_dataset.df["label"] == 0).sum()
    pos_weight = torch.tensor([num_neg / num_pos]).to(DEVICE)
    print(f"pos_weight: {pos_weight.item():.2f}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=DEVICE == "cuda")
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=DEVICE == "cuda")

    model     = TheftDetectionLSTM().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS)

    best_f1        = 0.0
    patience_count = 0

    print(f"\n{'Ep':>4} | {'Train Loss':>10} | {'Val Loss':>9} | {'Acc':>6} | {'Prec':>6} | {'Rec':>6} | {'F1':>6} | {'LR':>9}")
    print("-" * 75)

    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.unsqueeze(1).to(DEVICE)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            optimizer.step()
            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)
        val_loss, acc, prec, rec, f1, _, _, _ = evaluate(model, val_loader, criterion, threshold=0.50)
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        flag = ""
        if f1 > best_f1:
            best_f1        = f1
            patience_count = 0
            torch.save({
                "epoch"           : epoch + 1,
                "model_state_dict": model.state_dict(),
                "val_loss"        : val_loss,
                "val_f1"          : f1,
                "input_size"      : KEYPOINT_DIM,
                "hidden_size"     : HIDDEN_SIZE,
                "num_layers"      : NUM_LAYERS,
                "window_size"     : WINDOW_SIZE,
            }, MODEL_SAVE_PATH)
            flag = "  saved"
        else:
            patience_count += 1
            if patience_count >= EARLY_STOP_PAT:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        print(f"{epoch+1:>4} | {train_loss:>10.4f} | {val_loss:>9.4f} | "
              f"{acc:>5.1%} | {prec:>5.1%} | {rec:>5.1%} | {f1:>5.1%} | {current_lr:>9.2e}{flag}")

    print("\nLoading best model for threshold tuning...")
    ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])

    _, _, _, _, _, _, all_labels, all_probs = evaluate(model, val_loader, criterion, threshold=0.50)
    best_thresh, _ = find_best_threshold(all_probs, all_labels)

    ckpt["threshold"] = best_thresh
    torch.save(ckpt, MODEL_SAVE_PATH)

    _, acc, prec, rec, f1, preds, labels, _ = evaluate(model, val_loader, criterion, threshold=best_thresh)

    print(f"\nFinal results  (threshold={best_thresh:.2f})")
    print(f"  Acc: {acc:.4f}  Prec: {prec:.4f}  Rec: {rec:.4f}  F1: {f1:.4f}")

    cm = confusion_matrix(labels, preds)
    print(f"\nConfusion matrix:")
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

    print("\nClassification report:")
    print(classification_report(labels, preds, target_names=["Normal", "Theft"]))
    print(f"\nModel saved -> {MODEL_SAVE_PATH}  (threshold={best_thresh:.2f})")


train_model()

Device: cpu  |  params: 19,537  |  window: 30

Loading data...
  N:\Final Preparation\FULL_DATASET\new\training  ->  1181 samples  |  theft: 256  normal: 925
  N:\Final Preparation\FULL_DATASET\new\validation  ->  180 samples  |  theft: 36  normal: 144
pos_weight: 3.61

  Ep | Train Loss |  Val Loss |    Acc |   Prec |    Rec |     F1 |        LR
---------------------------------------------------------------------------
   1 |     1.1412 |    1.0627 | 50.0% | 23.5% | 66.7% | 34.8% |  2.00e-04  saved
   2 |     1.0663 |    0.9912 | 58.3% | 31.4% | 91.7% | 46.8% |  3.00e-04  saved
   3 |     1.0227 |    0.9344 | 62.8% | 33.3% | 86.1% | 48.1% |  4.00e-04  saved
   4 |     0.9458 |    0.9352 | 75.0% | 30.4% | 19.4% | 23.7% |  5.00e-04
   5 |     0.9104 |    0.7311 | 85.6% | 75.0% | 41.7% | 53.6% |  5.00e-04  saved
   6 |     0.9287 |    1.6594 | 30.0% | 22.2% | 100.0% | 36.4% |  5.00e-04
   7 |     0.9621 |    0.8845 | 51.7% | 29.3% | 100.0% | 45.3% |  4.99e-04
   8 |     0.8586 |    0.89

In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc,
)


# ── paths ─────────────────────────────────────────────────────────────────────
TEST_DIR        = r"N:\Final Preparation\FULL_DATASET\new\testing"
MODEL_SAVE_PATH = r"N:\Final Preparation\model\best_theft_lstm_model_v6.pth"
REPORT_SAVE_DIR = r"N:\Final Preparation\FULL_DATASET\new\testing\results"

CSV_FILE        = "labels.csv"
POSITIVE_FOLDER = r"positive\standardized_vedio\frame\lstm_dataset"
NEGATIVE_FOLDER = r"negitive\standardized_vedio\frame\lstm_dataset"

WINDOW_SIZE  = 30
KEYPOINT_DIM = 51
HIDDEN_SIZE  = 48
NUM_LAYERS   = 1
DROPOUT      = 0.6
BATCH_SIZE   = 8
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"


# ── dataset ───────────────────────────────────────────────────────────────────
class PoseSequenceDataset(Dataset):

    def __init__(self, root_dir):
        import pandas as pd
        csv_path = os.path.join(root_dir, CSV_FILE)
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"labels.csv not found: {csv_path}")
        self.df   = pd.read_csv(csv_path)
        self.root = root_dir
        pos = (self.df["label"] == 1).sum()
        neg = (self.df["label"] == 0).sum()
        print(f"  Test set  ->  {len(self.df)} samples  |  theft: {pos}  normal: {neg}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        fname     = row["filename"]
        label     = int(row["label"])
        subfolder = POSITIVE_FOLDER if label == 1 else NEGATIVE_FOLDER
        path      = os.path.join(self.root, subfolder, fname)
        data      = np.load(path).astype(np.float32)
        return (
            torch.tensor(data,  dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )


# ── model (identical to training) ─────────────────────────────────────────────
class TheftDetectionLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.lstm    = nn.LSTM(input_size=KEYPOINT_DIM, hidden_size=HIDDEN_SIZE,
                               num_layers=NUM_LAYERS, batch_first=True, dropout=0.0)
        self.bn      = nn.BatchNorm1d(HIDDEN_SIZE)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        return self.fc(out)


# ── plots ─────────────────────────────────────────────────────────────────────
def save_confusion_matrix(cm, save_path):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "Theft"],
                yticklabels=["Normal", "Theft"], ax=ax)
    ax.set_xlabel("Predicted",  fontsize=12)
    ax.set_ylabel("Actual",     fontsize=12)
    ax.set_title("Test — Confusion Matrix", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  saved -> {save_path}")


def save_roc_curve(labels, probs, save_path):
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc     = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, color="#2563eb", lw=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate",  fontsize=12)
    ax.set_title("Test — ROC Curve", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  saved -> {save_path}")


def save_metrics_bar(acc, prec, rec, f1, save_path):
    names  = ["Accuracy", "Precision", "Recall", "F1 Score"]
    values = [acc*100, prec*100, rec*100, f1*100]
    colors = ["#2563eb", "#16a34a", "#dc2626", "#9333ea"]
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(names, values, color=colors, width=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.8,
                f"{val:.1f}%", ha="center", va="bottom",
                fontsize=11, fontweight="bold")
    ax.set_ylim(0, 110)
    ax.set_ylabel("Score (%)", fontsize=12)
    ax.set_title("Test Metrics", fontsize=13, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  saved -> {save_path}")


# ── run test evaluation ───────────────────────────────────────────────────────
def run_test():
    os.makedirs(REPORT_SAVE_DIR, exist_ok=True)

    # load checkpoint
    print(f"Loading model  ->  {MODEL_SAVE_PATH}")
    ckpt      = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
    threshold = ckpt.get("threshold", 0.7)
    print(f"  best val F1 : {ckpt.get('val_f1', 0):.4f}  |  threshold: {threshold:.2f}  |  device: {DEVICE}")

    model = TheftDetectionLSTM().to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # load test data
    print("\nLoading test data...")
    test_dataset = PoseSequenceDataset(TEST_DIR)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # inference
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE)
            logits  = model(X_batch)
            probs   = torch.sigmoid(logits).squeeze(1)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.long().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs >= threshold).astype(int)

    # metrics
    acc     = accuracy_score (all_labels, preds)
    prec    = precision_score(all_labels, preds, zero_division=0)
    rec     = recall_score   (all_labels, preds, zero_division=0)
    f1      = f1_score       (all_labels, preds, zero_division=0)
    cm      = confusion_matrix(all_labels, preds)
    roc_auc = auc(*roc_curve(all_labels, all_probs)[:2])

    # print results
    print(f"\n{'='*50}")
    print(f"  TEST RESULTS  (threshold={threshold:.2f})")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  Precision : {prec*100:.2f}%")
    print(f"  Recall    : {rec*100:.2f}%")
    print(f"  F1 Score  : {f1*100:.2f}%")
    print(f"  ROC-AUC   : {roc_auc:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"    FN={cm[1,0]}  TP={cm[1,1]}")
    print(f"\n  Classification Report:")
    print(classification_report(all_labels, preds, target_names=["Normal", "Theft"]))

    # save plots
    print("Saving plots...")
    save_confusion_matrix(cm,
        os.path.join(REPORT_SAVE_DIR, "test_confusion_matrix.png"))
    save_roc_curve(all_labels, all_probs,
        os.path.join(REPORT_SAVE_DIR, "test_roc_curve.png"))
    save_metrics_bar(acc, prec, rec, f1,
        os.path.join(REPORT_SAVE_DIR, "test_metrics_bar.png"))

    print(f"\nAll plots saved -> {REPORT_SAVE_DIR}")


run_test()

Loading model  ->  N:\Final Preparation\model\best_theft_lstm_model_v6.pth
  best val F1 : 0.8732  |  threshold: 0.45  |  device: cpu

Loading test data...
  Test set  ->  332 samples  |  theft: 43  normal: 289

  TEST RESULTS  (threshold=0.45)
  Accuracy  : 88.86%
  Precision : 55.00%
  Recall    : 76.74%
  F1 Score  : 64.08%
  ROC-AUC   : 0.9420

  Confusion Matrix:
    TN=262  FP=27
    FN=10  TP=33

  Classification Report:
              precision    recall  f1-score   support

      Normal       0.96      0.91      0.93       289
       Theft       0.55      0.77      0.64        43

    accuracy                           0.89       332
   macro avg       0.76      0.84      0.79       332
weighted avg       0.91      0.89      0.90       332

Saving plots...
  saved -> N:\Final Preparation\FULL_DATASET\new\testing\results\test_confusion_matrix.png
  saved -> N:\Final Preparation\FULL_DATASET\new\testing\results\test_roc_curve.png
  saved -> N:\Final Preparation\FULL_DATASET\new\t